In [ ]:
import pandas as pd
import requests, io

## 有２種 API 的 URL

In [ ]:
# csv
# api_url = "https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=datacreationdate%20desc&format=CSV"

# json
api_url = "https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=846e44e1-8cc5-4893-ad87-c79d2d383706&limit=1000&sort=datacreationdate%20desc&format=JSON"

## 避開SSL認證
- verify=False

In [ ]:
resp = requests.get(api_url, verify=False)
print(resp)

### .read_csv() 或 .read_json() 

In [ ]:
# csv
# df = pd.read_csv(io.StringIO(resp.text))  # 因受限於資源分配，每日呼叫API的次數不可大於5000次。

# json
df = pd.read_json(io.StringIO(resp.text))
df

## 清理資料、移除空值、移除重複

In [ ]:
## 查看資料結構→ 空值、筆數、型態
df.info()

In [ ]:
df.describe()

In [ ]:
# 查看重複值
# df.duplicated()
# df[df.duplicated(subset=["site", "datacreationdate"])]

# df.drop_duplicates()
df.drop_duplicates(subset=["site", "datacreationdate"])

### .dropna() 移除空值

In [ ]:
df.drop_duplicates(subset=["site", "datacreationdate"]).dropna()

In [ ]:
df1 = df.drop_duplicates(subset=["site", "datacreationdate"]).dropna()
df1

## SQLite3 與 MySQL(PyMySQL)

- 在 CREATE 語法上的差異

|CREATE 語法|SQLite3|MySQL(PyMySQL)|
| ------------- |:-------------|:-------------|
|ID|id `integer` primary key **autoincrement**|id `int` primary key **auto_increment**|
|自動遞增|autoincrement|auto_increment(有底線)|
|整數|integer|int|
|文字、字串|text|VARCHAR(N)：需要指定長度N|
|日期時間格式|text|datatime|
|unique(複合唯一約束)|寫法（二選一）|寫法（三選一）|
|寫法 A：使用 CONSTRAINT 關鍵字（推薦，最標準）|CONSTRAINT uq_規則名字 UNIQUE (欄位1, 欄位2)|CONSTRAINT uq_規則名字 UNIQUE (欄位1, 欄位2)||
|寫法 B：不具名寫法（系統自動命名）|UNIQUE (欄位1, 欄位2)|UNIQUE (欄位1, 欄位2)|
|寫法 C：具名唯一鍵UNIQUE [KEY] [規則名字] (欄位)|❌|**UNIQUE KEY** uq_規則名字 UNIQUE (欄位1, 欄位2)|

- 在 INSERT 語法上的差異
    > SQLite3 用 `insert or ignore`：遇到重複資料，它會**優雅地跳過**，程式繼續執行。  
    > MySQL(PyMySQL) 用 `insert ignore`：遇到重複資料，它會**優雅地跳過**，程式繼續執行。

|INSERT 語法|SQLite3|MySQL(PyMySQL)|
|:-------------:|:-------------:|:-------------:|
|佔位符符號|使用問號 ? 作為佔位符|使用 %s 作為佔位符|
|寫法|`insert or ignore` into 資料表名稱 (欄位名稱) **values(?)**|`insert ignore` into 資料表名稱 (欄位名稱) **values(%s)**|

## sqlite3 建立資料庫

In [ ]:
import sqlite3

In [ ]:
sqlstr = '''
create table if not exists data(
id integer primary key autoincrement,
site text,
county text,
pm25 integer,
datacreationdate text,
itemunit text,
unique(site, datacreationdate)
)
'''

In [ ]:
conn = sqlite3.connect("pm25.db")
cursor = conn.cursor()
conn, cursor

In [ ]:
cursor.execute(sqlstr)
conn.commit()

### 插入資料
- or ignore
    - 忽略重複資料


In [ ]:
sqlstr = "insert or ignore into data (site,county,pm25,datacreationdate,itemunit)\
    values(?, ?, ?, ?, ?)"

In [ ]:
# df1.values
df1.values.tolist()

https://inloop.github.io/sqlite-viewer/

In [ ]:
# 插入多筆 .executemany
cursor.executemany(sqlstr, df1.values.tolist())
conn.commit()

In [ ]:
# cursor.rowcount 查看更新幾筆資料
cursor.rowcount

In [ ]:
conn.close()